## Tools
Models can request to call tools that perform tasks such as fetching the data from the database, searching the web, or running the code. Tools are pairings of
    
1. A schema, including the name of the tool, a discription, and argument definition ( often a JSON schema )
2. A function or coroutine to execute

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [8]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3-32b")

In [9]:
from langchain.tools import tool
import requests

@tool
def get_weather(location:str)->str:
    """Get current weather at a location"""
    weather_api=os.getenv("WEATHER_API_KEY")
    url = f"http://api.weatherstack.com/current?access_key={weather_api}&query={location}"
    response = requests.get(url)
    return response.json()

model_with_tools=model.bind_tools([get_weather])

In [10]:
response=model_with_tools.invoke("What is the wheather like in Hyderabad")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Hyderabad. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. The user spelled Hyderabad with an \'e\' instead of an \'a\', but I should use the correct spelling. The required parameter is location, so I\'ll input "Hyderabad" as the value. I need to make sure the JSON is correctly formatted with the function name and arguments. Let me structure the tool_call accordingly.\n', 'tool_calls': [{'id': 'tpw2vp50r', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 154, 'total_tokens': 277, 'completion_time': 0.226336877, 'completion_tokens_details': {'reasoning_tokens': 98}, 'prompt_time': 0.00737775, 'prompt_tokens_details': None, 'queue_time': 0.05791747, 'total_time': 0.233714627}, 'model_name': 'qwen

In [11]:
tool_call = response.tool_calls[0]
result = get_weather.invoke(tool_call["args"])

print(result)

{'request': {'type': 'City', 'query': 'Hyderabad, India', 'language': 'en', 'unit': 'm'}, 'location': {'name': 'Hyderabad', 'country': 'India', 'region': 'Telangana', 'lat': '17.375', 'lon': '78.474', 'timezone_id': 'Asia/Kolkata', 'localtime': '2026-07-08 18:50', 'localtime_epoch': 1783536600, 'utc_offset': '5.50'}, 'current': {'observation_time': '01:20 PM', 'temperature': 31, 'weather_code': 143, 'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0006_mist.png'], 'weather_descriptions': ['Haze'], 'astro': {'sunrise': '05:48 AM', 'sunset': '06:55 PM', 'moonrise': 'No moonrise', 'moonset': '12:43 PM', 'moon_phase': 'Waning Crescent', 'moon_illumination': 48}, 'air_quality': {'co': '306', 'no2': '10.7', 'o3': '68', 'so2': '3.7', 'pm2_5': '8.5', 'pm10': '11.2', 'us-epa-index': '1', 'gb-defra-index': '1'}, 'wind_speed': 26, 'wind_degree': 251, 'wind_dir': 'WSW', 'pressure': 1005, 'precip': 0, 'humidity': 55, 'cloudcover': 75, 'feelslike': 35, 'uv_index

In [12]:
for tool_call in response.tool_calls:
    #view the tool calls 
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

Tool: get_weather
Tool: {'location': 'Hyderabad'}


### Tool Execution loops

In [13]:
# step 1: model generates tool calls
messages=[{"role":"user", "content":"What's the weather like in Hyderabad today?"}]
ai_msgs=model_with_tools.invoke(messages)
messages.append(ai_msgs)

# step 2: Execute tool and collect the results
for tool_call in ai_msgs.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: pass result back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Hyderabad is **31°C** with **Haze**. Here's a detailed breakdown:  
- **Wind**: 26 km/h from WSW  
- **Humidity**: 55%  
- **Feels Like**: 35°C  
- **UV Index**: 0 (No UV radiation)  
- **Visibility**: 5 km  
- **Air Quality**: Good (US-EPA Index: 1)  

The sky is partly cloudy with 75% cloud cover. Sunset today will be at **6:55 PM**.


In [14]:
messages

[{'role': 'user', 'content': "What's the weather like in Hyderabad today?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Hyderabad today. I need to use the get_weather function. The function requires a location parameter. Hyderabad is the location here. So I should call get_weather with location set to "Hyderabad". Let me make sure there are no other parameters needed. The function only asks for location, so that\'s all I need. Alright, time to format the tool call correctly.\n', 'tool_calls': [{'id': 'fpymewpty', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 155, 'total_tokens': 264, 'completion_time': 0.173050295, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.006812128, 'prompt_tokens_details': None, 'queue_time': 0.056208661, 'total_time': 0.179862423}, 'model_n